# Analysis figure suite (editable)

Interactive driver for `eye_tracking_system_tools.analysis`.

**Two modes**
1. **Paper events** — load the frozen Fig‑2j event pickle (exact published N / filters) and re-export 2c–2j (2f from archived pickle).
2. **Remount** — re-detect saccades from analyzed block CSVs via a YAML registry.

Outputs go under `outputs/<run>/{figures,metadata}/`. Empty `TAG` overwrites `phase2_latest`; set `TAG` to keep a snapshot.

Edit the **Parameters** cell, then run the rest top-to-bottom.

Requires: `PYTHONPATH` including `src` (set in the setup cell), and the `eye_repo_mac` (or equivalent) env.

## 0. Setup

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Repo root = parent of this notebook's folder (development/ → PETS/)
REPO = Path.cwd()
if (REPO / "src" / "eye_tracking_system_tools").is_dir():
    pass
elif (REPO.parent / "src" / "eye_tracking_system_tools").is_dir():
    REPO = REPO.parent
else:
    # Fallback: walk up
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.pipeline import (
    build_event_tables,
    event_tables_from_saccade_angles_pickle,
    run_figure_exports,
)
from eye_tracking_system_tools.analysis.block_registry import load_registry, registry_summary
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir

print("REPO:", REPO)

REPO: /Users/nimi/Projects/PETS


## 1. Parameters (edit me)

Toggle `MODE`, paths, and which figures to export. You can also override individual YAML keys via `PARAM_OVERRIDES`.

In [ ]:
# --- Mode ---
# "paper_events" = frozen Fig_2_j event pickle (closest to published numbers)
# "remount"      = re-detect from analyzed blocks on disk
MODE = "remount"  # or "paper_events"

# --- Paths ---
PARAMS_YAML = REPO / "configs" / "analysis_params.yaml"
REGISTRY_YAML = REPO / "configs" / "paper_blocks.yaml"  # used when MODE == "remount"
EVENT_PICKLE = (
    REPO
    / "src/eye_tracking_system_tools/figures/reproduction/main_figures"
    / "Fig_2_j/saccade_angles_data.pkl"
)
OUT_ROOT = REPO / "outputs"
TAG = ""  # empty → phase2_latest (overwrite); e.g. "paper_events_v2"
_run = resolve_run_dir(OUT_ROOT, TAG or None)
OUT_DIR = _run.run_dir
FIGURES_DIR = _run.figures_dir
METADATA_DIR = _run.metadata_dir

# Which figures to export (2f from archived pickle in paper_events; live in remount)
FIGURES = ["2c", "2d", "2e", "2f", "2g", "2h", "2i", "2j"]

# Optional overrides merged on top of PARAMS_YAML (edit freely)
PARAM_OVERRIDES = {
    # "saccade": {"speed_threshold_deg_per_frame": 0.8, "min_net_disp_deg": 0.5},
    # "binocular": {"sync_diff_ms": 34.0},
    # "main_sequence": {"velocity_unit": "deg/sec", "frame_rate_fps": 60.0, "plot_animals": ["PV_106"]},
    # "figure_2f": {
    #     "event_mode": "monocular",
    #     "require_head_stationary": True,
    #     "exclude_animals": ["PV_62", "PV_57"],
    # },
}

print("MODE:", MODE)
print("OUT:", OUT_DIR)
print("  figures:", FIGURES_DIR)
print("  metadata:", METADATA_DIR)

MODE: remount
OUT: /Users/nimi/Projects/PETS/outputs/phase2_latest
  figures: /Users/nimi/Projects/PETS/outputs/phase2_latest/figures
  metadata: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata


## 2. Load params + build event tables

In [3]:
def deep_merge(base: dict, over: dict) -> dict:
    out = dict(base)
    for k, v in over.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = v
    return out

params = deep_merge(load_params_yaml(PARAMS_YAML), PARAM_OVERRIDES)
print("Loaded params keys:", list(params.keys()))
print(yaml.safe_dump({k: params[k] for k in params if k in ("saccade", "binocular", "figure_2f")}, sort_keys=False))

Loaded params keys: ['saccade', 'binocular', 'main_sequence', 'figure_2f', 'figure_2g', 'figure_2h', 'figure_2i']
saccade:
  speed_threshold_deg_per_frame: 0.8
  directional_delta_threshold_deg: 90.0
  min_subsaccade_samples: 2
  min_net_disp_deg: 0.5
  speed_profile: true
binocular:
  sync_diff_ms: 34.0
figure_2f:
  iqr_multiplier: 60.0
  bins: 60
  macro_range:
  - 0.0
  - 0.5
  micro_range:
  - 0.0
  - 0.1
  macro_tick_list:
  - 0.0
  - 0.25
  - 0.5
  micro_tick_list:
  - 0.0
  - 0.05
  - 0.1
  event_mode: monocular
  require_head_stationary: true
  exclude_animals:
  - PV_62
  - PV_57
  contra_event_window_ms: 100.0
  contra_sample_ms: 51.0
  also_export_s3: true



In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "paper_events":
    tables = event_tables_from_saccade_angles_pickle(EVENT_PICKLE, params=params)
    print(f"Loaded paper events from {EVENT_PICKLE}")
elif MODE == "remount":
    specs = load_registry(REGISTRY_YAML)
    print(registry_summary(specs))
    tables = build_event_tables(specs, params=params)
else:
    raise ValueError(f"Unknown MODE={MODE!r}")

print(
    f"events: all={len(tables.all_saccades)} "
    f"synced_rows={len(tables.synced)} non_synced={len(tables.non_synced)}"
)
print("animals:", sorted(tables.all_saccades["animal"].astype(str).unique()))

{'n_blocks': 23, 'animals': ['PV_106', 'PV_126', 'PV_143', 'PV_57', 'PV_62'], 'blocks': [{'animal': 'PV_106', 'block_num': '008', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_106/2025_08_06/block_008'}, {'animal': 'PV_106', 'block_num': '009', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_106/2025_08_06/block_009'}, {'animal': 'PV_106', 'block_num': '010', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_106/2025_08_06/block_010'}, {'animal': 'PV_106', 'block_num': '011', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_106/2025_08_06/block_011'}, {'animal': 'PV_106', 'block_num': '012', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_106/2025_08_06/block_012'}, {'animal': 'PV_143', 'block_num': '001', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_143/2025_08_25/block_001'}, {'animal': 'PV_143', 'block_num': '002', 'block_path': '/Volumes/Data-2/Nimrod/experiments/PV_143/2025_08_25/block_002'}, {'animal': 'PV_143', 'block_num': '003', 'block_path': '/

## 3. Inspect event column ranges

Use this to sanity-check units / filters before plotting.

- `peak_velocity` in event tables is **deg/frame** (paper convention).
- Fig 2e converts to **deg/ms** via `peak * (fps/1000)`.
- Fig 2c traces use **deg/sec**.

In [6]:
df = tables.all_saccades
cols = [
    "net_angular_disp", "overall_angle_deg", "peak_velocity",
    "length", "magnitude_raw_angular", "delta_phi", "delta_theta",
]
rows = []
for c in cols:
    if c not in df.columns:
        continue
    s = pd.to_numeric(df[c], errors="coerce")
    rows.append(
        dict(col=c, min=s.min(), max=s.max(), median=s.median(), nan_pct=100 * s.isna().mean())
    )
display(pd.DataFrame(rows).round(4))

if "head_movement" in df.columns:
    print("head_movement value counts:")
    print(df["head_movement"].value_counts(dropna=False))

,col,min,max,median,nan_pct
0,net_angular_disp,0.5000,53.5902,2.9175,4.3648
1,overall_angle_deg,0.0094,359.9844,183.0724,4.3648
2,peak_velocity,0.8001,74.2634,2.2741,0.0000
3,length,1.0000,19.0000,2.0000,0.0000
4,magnitude_raw_angular,0.8011,104.8274,5.1241,0.0000
5,delta_phi,-40.9434,50.5415,-0.0016,4.1376
6,delta_theta,-37.5347,50.7937,-0.0416,4.3648


head_movement value counts:
head_movement
1.0    14776
NaN    11088
0.0    10220
Name: count, dtype: int64


## 4. Export figures

Writes PDFs under `FIGURES_DIR` and pickles / `*.meta.yaml` under `METADATA_DIR`.
In paper_events mode, Fig 2f is drawn from the archived nodowncast pickle into the run folder.

In [7]:
figs = list(FIGURES)
written = run_figure_exports(
    tables,
    OUT_DIR,
    figures=figs,
    include_archived_2f=(MODE == "paper_events"),
)
for name, path in written.items():
    if name.startswith("_"):
        continue
    print(f"{name}: {path}")

/Users/nimi/Projects/PETS/src/eye_tracking_system_tools/analysis/figures_2c_2e.py:142: RuntimeWarning: Mean of empty slice
  vel_center = np.nanmean(vel_stack, axis=0).astype(np.float32)


[2f] head_stationary filter: 19212 → 6996
[2f] event_mode=monocular kept=1893 skip_mode=4424 skip_profile=679 exclude=['PV_57', 'PV_62']
event_tables: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/event_tables.pkl
2c_2d: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/pos_vel_by_amp_bins_bundle.pkl
2e: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/amplitude_velocity_linear_fit_bundle.pkl
2f: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/figure_2f_nodowncast.pickle
2h: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/figure_2h.pickle
2i: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/figure_2i.pickle
2g_top: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/averaged_saccade_amplitude_angle_data.pkl
2g_bot: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/saccade_amplitude_difference_all_animals_data.pkl
2j: /Users/nimi/Projects/PETS/outputs/phase2_latest/metadata/saccade_angles_data.pkl


## 5. Optional: reproduction baseline + jitter

Copy archived reproduction PDFs into this run's `figures/repro_baseline/`, or seed/plot jitter epochs (see `configs/jitter_mount_blocks.yaml`).

In [ ]:
from eye_tracking_system_tools.analysis.repro_baseline import run_repro_baseline
from eye_tracking_system_tools.analysis.run_layout import RunDirs

# Uncomment to rebuild baseline PDFs into this run (can take a few minutes):
# run_repro_baseline(RunDirs(OUT_DIR, FIGURES_DIR, METADATA_DIR), repo=REPO)

print("Baseline target:", FIGURES_DIR / "repro_baseline")
print("Jitter CLI example:")
print(
    "  PYTHONPATH=src python -m eye_tracking_system_tools.analysis.jitter_epochs "
    "--registry configs/jitter_mount_blocks.yaml --out-root outputs --tag '' --pick --plot"
)

## 6. Quick preview of exported PDFs

In [10]:
import os

from IPython.display import display, FileLink, Markdown

# FileLink resolves its argument against the *kernel* working directory, which is
# development/ when the notebook is launched from its own folder — not REPO.
CWD = Path.cwd().resolve()

pdfs = sorted(FIGURES_DIR.rglob("*.pdf"))
print(f"{len(pdfs)} PDFs under {FIGURES_DIR}")
for p in pdfs:
    p = p.resolve()
    rel = os.path.relpath(p, CWD)
    if os.path.exists(rel):
        display(FileLink(rel))
    else:
        display(Markdown(f"[{p.name}]({p.as_uri()})"))

11 PDFs under /Users/nimi/Projects/PETS/outputs/phase2_latest/figures


/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/fig_2g_bot.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/fig_2g_top.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_2c.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_2d.pdf

...

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_2f_colorbar.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_2h.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_2i.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_2j.pdf

/Users/nimi/Projects/PETS/outputs/phase2_latest/figures/figure_S3.pdf

## 7. Compare remount vs paper event Ns (optional)

Run after you have both `outputs/phase2_paper_figures` (remount) and this notebook's paper-events export — or point `REMOUNT_EVENTS` at a remount `event_tables.pkl`.

In [ ]:
import pickle

REMOUNT_EVENTS = REPO / "outputs" / "phase2_paper_figures" / "event_tables.pkl"
# also try organized layout:
if not REMOUNT_EVENTS.exists():
    alt = REPO / "outputs" / "phase2_paper_remount" / "metadata" / "event_tables.pkl"
    if alt.exists():
        REMOUNT_EVENTS = alt

paper_n = len(tables.all_saccades)
print(f"this run all_saccades N = {paper_n}")

if REMOUNT_EVENTS.exists():
    with open(REMOUNT_EVENTS, "rb") as f:
        rem = pickle.load(f)
    print(f"remount all_saccades N = {len(rem['all_saccades'])}")
    print(f"Δ = {len(rem['all_saccades']) - paper_n}")
else:
    print(f"No remount events at {REMOUNT_EVENTS} (skip)")

## Notes

| Knob | Where |
|------|--------|
| Saccade detection | `params['saccade']` / `configs/analysis_params.yaml` |
| L/R pairing window | `params['binocular']['sync_diff_ms']` |
| Fig 2c–2e | `params['main_sequence']` (`plot_animals`, `bin_cmap`) |
| Fig 2f filters (R3-1) | `params['figure_2f']` — monocular, head-stationary, animal exclusions |
| Block list | `configs/paper_blocks.yaml` (remount) |
| Jitter mounts | `configs/jitter_mount_blocks.yaml` |
| Eye CSV choice | always prefers `*raw_verified*`; recorded in `*.meta.yaml` |

CLI equivalents:

```bash
# Paper event set → outputs/phase2_paper_events/{figures,metadata}/
PYTHONPATH=src python -m eye_tracking_system_tools.analysis \
  --event-pickle src/eye_tracking_system_tools/figures/reproduction/main_figures/Fig_2_j/saccade_angles_data.pkl \
  --params configs/analysis_params.yaml \
  --out-root outputs --tag paper_events

# Remount from Data-2
PYTHONPATH=src python -m eye_tracking_system_tools.analysis \
  --registry configs/paper_blocks.yaml \
  --params configs/analysis_params.yaml \
  --out-root outputs --tag paper_remount

# Overwrite working run
PYTHONPATH=src python -m eye_tracking_system_tools.analysis \
  --event-pickle ... --out-root outputs --tag ""
```